
# UNet Fine-Tuning for Lake Segmentation — **RGB from Planet 8-Band**

This notebook trains a **UNet** on **RGB** channels extracted from **Planet 8-band** GeoTIFF tiles (size **512×512**).  
We use **ImageNet-pretrained** encoders (ResNet, EfficientNet, MobileNet, …) so transfer learning works as intended.

**What it does**
- Randomly sample **1,000** image/label pairs by matching filenames.
- Extract **RGB** from Planet 8-band tiles (defaults to **(6,4,2) = (Red, Green, Blue)** for SuperDove SR).
- Train/Val split (80/20).
- Train UNet across multiple **backbones** using **pretrained(ImageNet)** weights.
- **Loss:** BCE + Soft Dice (good for class imbalance).
- **Metrics:** IoU, Precision, Recall, F1, Accuracy on **train + val**, logged to CSV.
- Save **best-train-IoU** checkpoint per backbone.

> If your band ordering is different, **change `RGB_BANDS`** below.


## Dependencies

In [ ]:

# If needed, uncomment to install:
# %pip install -q torch torchvision timm segmentation-models-pytorch rasterio albumentations pandas scikit-image


## Configuration

In [10]:

from pathlib import Path
import random

# === REQUIRED: Paths ===
IMAGES_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/images")  # change to your imagery folder
LABELS_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/labels")  # change to your label folder (same filenames)

# === PlanetScope SuperDove SR default band mapping (1-indexed) ===
RGB_BANDS = (3, 2, 1)  # (R, G, B) 1-indexed

# === Outputs ===
OUTPUT_DIR = Path("./training_output_rgb/subset_750")
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# === Training config ===
NUM_SAMPLES = 750
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-3

BACKBONES = ['mit_b0', 'mit_b1', 'mit_b2', 'mit_b3', 'mit_b4', 'mit_b5']

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [2]:

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")


Found 750 pairs.
Train: 600, Val: 150



## Dataset & Dataloaders (RGB extraction)
- Reads 8-band imagery and **selects RGB channels** via `RGB_BANDS` (1-indexed).
- Per-image **min–max** normalization to [0,1].
- Light flips for augmentation.


In [1]:
import torch

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W), 1-indexed bands in file
    return arr

def select_rgb(x: np.ndarray, rgb_bands=(6,4,2)):
    idx = [b-1 for b in rgb_bands]
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i]); vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTilesRGB(Dataset):
    def __init__(self, pairs, rgb_bands=(6,4,2), aug=None, normalize=True):
        self.pairs = pairs
        self.rgb_bands = rgb_bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img8 = read_geotiff(img_path)          # (8,H,W)
        img = select_rgb(img8, self.rgb_bands) # (3,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)                  # (H,W)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        # Albumentations expects HWC
        img_hwc = np.transpose(img, (1,2,0))
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        # Back to CHW
        img = torch.from_numpy(np.transpose(img_hwc, (2,0,1))).float()
        lbl = torch.from_numpy(lbl_hwc[...,0]).float()
        return img, lbl

train_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])
val_aug   = A.Compose([])

train_ds = LakeTilesRGB(train_pairs, rgb_bands=RGB_BANDS, aug=None, normalize=True)
val_ds   = LakeTilesRGB(val_pairs,   rgb_bands=RGB_BANDS, aug=None,   normalize=True)

PIN_MEMORY = False

# --- Adapter from your PyTorch loaders to Detectron2 style ---

class D2WrappedDataset(Dataset):
    # Wrap (img, lbl) into Detectron2 dicts
    def __init__(self, base_ds):
        self.base_ds = base_ds
    def __len__(self):
        return len(self.base_ds)
    def __getitem__(self, idx):
        img, lbl = self.base_ds[idx]  # img: (3,H,W) float in [0,1]; lbl: (H,W)
        if isinstance(lbl, torch.Tensor):
            lbl_t = lbl.long()
        else:
            import numpy as np
            lbl_t = torch.from_numpy(lbl.astype(np.int64))
        lbl_t = (lbl_t > 0).long()  # ensure {0,1}
        return {'image': img.float(), 'sem_seg': lbl_t}

def d2_collate(batch):
    return batch  # list[dict]


train_loader_d2 = DataLoader(D2WrappedDataset(train_ds), batch_size=BATCH_SIZE,
                             shuffle=True, pin_memory=PIN_MEMORY, collate_fn=d2_collate)
val_loader_d2   = DataLoader(D2WrappedDataset(val_ds),   batch_size=BATCH_SIZE,
                             shuffle=False, pin_memory=PIN_MEMORY, collate_fn=d2_collate)

len(train_loader_d2.dataset), len(val_loader_d2.dataset)



(600, 150)

# 🧊 Mask2Former for Glacial Lakes (Semantic Segmentation)

This section adds a **Mask2Former** training pipeline (Detectron2) for **binary semantic segmentation** of glacial lakes.
It **reuses your existing `LakeTilesRGB` `train_ds`/`val_ds` and `DataLoader`s** so you don't need to rewire data.


In [ ]:
# --- Install notes (run on your machine/Colab) ---
# pip install --upgrade pip
# pip install 'git+https://github.com/facebookresearch/detectron2.git'
# pip install 'git+https://github.com/facebookresearch/Mask2Former.git'
# See: https://detectron2.readthedocs.io/en/latest/tutorials/install.html


In [ ]:
# --- Imports ---
import os, glob
from pathlib import Path
import torch
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import SemSegEvaluator, inference_on_dataset
from detectron2.utils.logger import setup_logger
from mask2former import add_maskformer2_config
setup_logger()
print('Torch:', torch.__version__)


### 🔌 Reuse your **PyTorch LakeTilesRGB** dataloaders

Wrap each `(img, lbl)` sample as a Detectron2 dict: `{'image': CHW float, 'sem_seg': HW long}`.


In [ ]:
# --- Config (Mask2Former semantic segmentation) ---
cfg = get_cfg()
add_maskformer2_config(cfg)
# Example config; adjust to your preferred Mask2Former backbone/config file available in your env:
cfg.merge_from_file('configs/ade20k-150/semantic-segmentation/mask2former_swin_tiny_IN21k_384_bs16_160k.yaml')

# IMPORTANT: images are float RGB in [0,1]; disable extra normalization.
cfg.MODEL.PIXEL_MEAN = [0.0, 0.0, 0.0]
cfg.MODEL.PIXEL_STD  = [1.0, 1.0, 1.0]

# We use external loaders, not D2 DatasetCatalog.
cfg.DATASETS.TRAIN = ()
cfg.DATASETS.TEST  = ()

# Binary segmentation: background + lake
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 2

cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 1e-4
cfg.SOLVER.MAX_ITER = 20000
cfg.SOLVER.WARMUP_ITERS = 500
cfg.SOLVER.STEPS = []

cfg.OUTPUT_DIR = './output_mask2former_semseg_rgb'
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
with open(os.path.join(cfg.OUTPUT_DIR, 'config.yaml'), 'w') as f:
    f.write(cfg.dump())
print('OUTPUT_DIR:', cfg.OUTPUT_DIR)


In [ ]:
# --- Trainer that consumes our external loaders ---
_EXTERNAL_TRAIN_LOADER = train_loader_d2
_EXTERNAL_VAL_LOADER   = val_loader_d2

class TorchLoaderMask2FormerTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name=None, output_folder=None):
        return SemSegEvaluator(dataset_name or 'glacial_val', distributed=False, output_dir=cfg.OUTPUT_DIR)
    @classmethod
    def build_train_loader(cls, cfg):
        return _EXTERNAL_TRAIN_LOADER
    @classmethod
    def build_test_loader(cls, cfg, dataset_name=None):
        return _EXTERNAL_VAL_LOADER


In [ ]:
# --- Train ---
trainer = TorchLoaderMask2FormerTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


In [ ]:
# --- Evaluate (mIoU) ---
evaluator = SemSegEvaluator('glacial_val', distributed=False, output_dir=cfg.OUTPUT_DIR)
res = inference_on_dataset(trainer.model, _EXTERNAL_VAL_LOADER, evaluator)
print('Validation metrics:', res)
